# Lab 2 — Prompting a Language Model for Clinical Text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angeloziletti/nlp-cas-dsmh/blob/main/notebooks/lab2_prompting_clinical_text.ipynb)

In this lab you direct a real language model to do clinical work — and you inspect it with a critical eye.

**How to use it**
- Run each cell in order (`Shift + Enter`).
- Read the output *and* the explanation around it — we discuss every result together.
- Cells marked **🔧 YOUR TURN** are for you: change one value, re-run, see what happens.
- This lab uses the **Gemini API** and your API key (from Lab 0).

**What you will do**
1. **Extract** medications and diagnoses from discharge summaries
2. Apply the **"only what's stated"** rule and see if the references provided are correct
3. **Classify** notes — with an *uncertain* escape hatch
4. Compare **zero-shot vs few-shot**
5. Use **chain-of-thought** and watch it change an answer
6. Get **structured JSON** output a program can use
7. **Inspect** model output for hallucination
8. Check results against a **gold-standard set** — a first taste of evaluation
9. Hold a short **multi-turn conversation**

*All data is synthetic. No real patient information is used — the rule from Block D.*

## Setup

Three cells. The first installs the Gemini SDK; the second loads your API key and connects; the third confirms it works.

**Before you run the second cell**, your Gemini API key must be available:

- **In Colab:** click the **🔑 key icon** in the left sidebar (*Secrets*), add a secret named exactly `GEMINI_API_KEY`, paste your key as the value, and switch on **Notebook access**.
- **Running locally:** set it as an environment variable before launching Jupyter, e.g. on macOS/Linux `export GEMINI_API_KEY=...`, on Windows PowerShell `$env:GEMINI_API_KEY = "..."`.

This is the same key you created in Lab 0. The key is never written into the notebook itself.

In [1]:
!pip install -q google-genai

In [2]:
import json
import os
import pandas as pd
from google import genai
from google.genai import types
from typing_extensions import TypedDict

# --- Read the Gemini API key (Colab Secrets, or local .env / env var) ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found. In Colab, store it as a Secret named "
        "GEMINI_API_KEY. Locally, copy .env.example to .env and paste your "
        "key there (or set the environment variable before launching Jupyter)."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-3.1-flash-lite"

# --- Course data lives in the GitHub repo ---
BASE_URL = "https://raw.githubusercontent.com/angeloziletti/nlp-cas-dsmh/main/data/"

print("Connected. Model:", MODEL)

Connected. Model: gemini-3.1-flash-lite


In [3]:
# Confirm the key works — a tiny test call.
test = client.models.generate_content(
    model=MODEL,
    contents="Reply with exactly these three words: setup is working",
)
print(test.text)

setup is working


If the cell above printed a short reply, your setup is working. If it raised an error, check that the secret is named exactly `GEMINI_API_KEY` and that *Notebook access* is switched on.

Now we define one small helper and load the clinical notes.

In [4]:
# A helper so the rest of the lab stays readable: send a prompt, get the text back.
def ask(prompt):
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

# Load the 12 synthetic discharge summaries (each comes with gold-standard answers).
notes = pd.read_json(BASE_URL + "lab2_notes.json").to_dict("records")
print(f"Loaded {len(notes)} synthetic discharge summaries.")
print("Example — note", notes[0]["id"], ":\n")
print(notes[0]["text"])

Loaded 12 synthetic discharge summaries.
Example — note N01 :

Mr. A, 64, admitted with an acute non-ST-elevation myocardial infarction. Underwent coronary angiography with stenting of the right coronary artery; recovery uncomplicated. Discharged on aspirin, ticagrelor, atorvastatin, bisoprolol and ramipril. Cardiac rehabilitation referral made. To be reviewed in the cardiology clinic in six weeks. GP to monitor blood pressure and renal function.


---
## Part 1 — Extraction: pulling structured facts from a note

Our first task is **extraction** — pulling specific facts out of free text. Here: every medication in a discharge summary.

Look at the prompt below. It is built from the **five ingredients** from the Block C lecture — *Role, Task, Context, Format* — each labelled in a comment.

In [5]:
note = notes[0]["text"]

prompt = f"""You are a clinical data assistant.                          # Role
Extract every medication mentioned in the discharge summary below.       # Task
Use only medications explicitly stated in the text; do not add anything. # Context (the grounding rule)
Return them as a comma-separated list, generic names only, lowercase.    # Format

Discharge summary:
{note}"""

print(ask(prompt))

aspirin, ticagrelor, atorvastatin, bisoprolol, ramipril


Compare the output to the note above — every medication, nothing invented. The grounding rule (*"only medications explicitly stated"*) and the explicit format are what make the result clean and predictable.

In [6]:
# 🔧 YOUR TURN: change the index to any note from 0 to 11 and re-run.
note_index = 3

chosen = notes[note_index]
prompt = f"""You are a clinical data assistant.
Extract every medication mentioned in the discharge summary below.
Use only medications explicitly stated in the text; do not add anything.
Return them as a comma-separated list, generic names only, lowercase.

Discharge summary:
{chosen["text"]}"""

print(f"Note {chosen['id']}:")
print(ask(prompt))

Note N04:
paracetamol, codeine, enoxaparin


---
## Part 2 — Watching the model hallucinate a citation

The first form of hallucination most people imagine — *"the model will invent a drug name or a dose that is not in the note"* — has largely been trained out of frontier models. Ask one for an antibiotic that the note never names and it will usually tell you it cannot say. That is real progress.

The more dangerous form is still very much alive: **fabricated citations.** The model is trained on the open web, which is full of guideline names, section numbers, PubMed IDs, journal titles, and direct quotes. When you ask it to source a clinical recommendation, it produces *something* — confidently, in the polished voice of a textbook. Most of the time the citation is real. Sometimes it is partly real (right paper, wrong section). Sometimes it is invented from whole cloth. **And the three cases are indistinguishable by surface inspection.**

To make this visceral, we ask the model to write an auditable clinical justification for each note. We demand five specific facts — guideline, URL, verbatim quote, landmark-trial PMID, first author and journal — and check the URL and PMID *programmatically*. **You check the rest.**

In [7]:
import re
import urllib.error
import urllib.request

URL_RE  = re.compile(r"https?://[^\s)\"<>\]]+")
PMID_RE = re.compile(r"\bPMID:?\s*(\d{4,9})\b", re.IGNORECASE)
DOI_RE  = re.compile(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+", re.IGNORECASE)


def head(url, timeout=8):
    req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception:
        return None


def verdict(code):
    if code == 200:        return "OK"
    if code in (301, 302): return f"redirect ({code})"
    if code == 403:        return f"{code} (paywall - unverifiable)"
    return f"broken ({code})" if code else "broken (no response)"


prompt_audit = """You are writing a clinical justification entry for the medical record. For full traceability, your response MUST include each of:
  1. The primary clinical guideline (name + year + recommendation number).
  2. A direct, working URL to the guideline.
  3. A verbatim quote, in quotation marks, of the supporting recommendation.
  4. The PubMed PMID or DOI of the original landmark trial that established this treatment.
  5. The first author and journal of that trial.

Be specific and concrete throughout.

Discharge summary:
"""


# Run the audit prompt on three notes. The failure modes vary across notes
# and across runs, so three is enough to make the pattern visible without
# burying you in output.
for nid in ["N01", "N04", "N08"]:
    note = next(n for n in notes if n["id"] == nid)
    print(f"=== Note {nid} ".ljust(78, "="))
    response = ask(prompt_audit + note["text"])
    print(response)
    print("\n--- machine-checkable claims ---")

    urls  = [u.rstrip(".,;)") for u in dict.fromkeys(URL_RE.findall(response))]
    pmids = list(dict.fromkeys(PMID_RE.findall(response)))
    dois  = [d.rstrip(".,;)") for d in dict.fromkeys(DOI_RE.findall(response))]

    for u in urls:
        print(f"  URL   {verdict(head(u)):30s}  {u}")
    for pmid in pmids:
        purl = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        print(f"  PMID  {verdict(head(purl)):30s}  {purl}")
    for doi in dois:
        durl = f"https://doi.org/{doi}"
        print(f"  DOI   {verdict(head(durl)):30s}  {durl}")
    if not (urls or pmids or dois):
        print("  [no URLs/PMIDs/DOIs extracted]")
    print()

=== Note N01 =================================================================
**Clinical Justification: Dual Antiplatelet Therapy (DAPT) Post-NSTEMI/PCI**

**1. Primary Clinical Guideline:**
ESC Guidelines for the management of acute coronary syndromes (2023), Recommendation 7.4.2.

**2. Guideline URL:**
https://academic.oup.com/eurheartj/article/44/38/3720/7243916

**3. Verbatim Quote:**
"In patients with NSTE-ACS undergoing PCI, DAPT consisting of a P2Y12 receptor inhibitor—preferably prasugrel or ticagrelor—in addition to aspirin is recommended for 12 months, unless there are contraindications such as excessive risk of bleeding."

**4. Landmark Trial PMID:**
PMID: 20201518

**5. First Author and Journal:**
Wallentin L; *New England Journal of Medicine* (The PLATO Trial).

***

**Clinical Rationale:**
Mr. A, a 64-year-old male, underwent successful coronary stenting for an NSTEMI. Consistent with the 2023 ESC guidelines, ticagrelor is indicated as the preferred P2Y12 inhibitor over 

**Now act like an auditor.** The block above was designed to make manual verification trivial — every model response includes a **clickable URL**, a **PubMed PMID**, and an **explicit verbatim quote**. The programmatic checks have already flagged any URL or PMID that **fails to resolve** (404, 403). That is the cheap half.

**Your job is the part a HEAD check cannot see:**

1. **For every URL that resolved (200)**, open it and search the document for the *recommendation number* and the *verbatim quote* the model produced. Is the text actually there, word for word? Does it apply to the exact situation in *this* patient's note — same surgery, same age group, same severity?
2. **For every PMID that resolved**, click through. A green `200` only tells you that the number exists in PubMed — **not** that it points to the paper the model claimed. Open the paper: is the title about the right intervention? Is the first author the one the model named? Models will sometimes pair a real-author claim with a real-but-unrelated PMID, which is the citation hallucination most likely to slip through review.
3. **Watch for hedging language.** Phrases like *"While the patient underwent X, the guideline for Y consistently aligns..."* are a tell — the model is justifying a substitution. Treat them as red flags worth digging into.
4. **An all-green programmatic check is necessary but not sufficient.** The most dangerous failures are real URLs leading to the wrong section, real PMIDs pointing to a paper about a different intervention. They pass any automated check; only a clinician's verification catches them.

You may find a response that is entirely correct, and one that is full of subtle errors. **That is the lesson.** A clinician auditing a model's output cannot know in advance which class a given response falls into — only by checking. Modern clinical hallucination is not a wrong drug name. It is a fabricated PubMed ID, a section number in the wrong chapter, a one-character typo in a URL, or a verbatim quote that is real but about the wrong surgery. Each one slips past a tired reader.

> **A common problem in clinical AI.** *Automatically* verifying that the verbatim quote appears in the cited source is genuinely hard. Clinical guidelines live as PDFs, behind paywalls, with JavaScript-rendered HTML — a simple "fetch and grep" works for some bodies (NICE) and fails for others (GOLD, ESC, AHA). Going further than the lab does would require PDF parsing, fuzzy matching, and possibly a second model acting as a verifier.

Now look at what the **grounded** version produces instead.

In [9]:
note4 = next(n for n in notes if n["id"] == "N04")

prompt_grounded = f"""Identify any clinical guideline EXPLICITLY referenced in the discharge summary below.
Use ONLY guideline names that appear in the text of the note.
If no guideline is explicitly cited in the note, reply exactly: not specified.

Discharge summary:
{note4["text"]}"""

print(ask(prompt_grounded))

not specified


**`not specified.`** — the right answer when the source does not contain the cited information. It is ungraceful, machine-parseable, and unmistakable. A downstream system can route on it; a tired clinician cannot fail to notice it.

**The rule from Block D.** When you need a model to source a clinical recommendation, two things must be in the prompt: a **grounding rule** (*"use only what is in this note"*) and an **escape hatch** (*"reply 'not specified' if it is not there"*). Without the escape hatch the model is being asked the impossible — to produce a citation that does not exist — and it will oblige with one that *looks* real. With the escape hatch, the model's uncertainty becomes a single, unambiguous string downstream code can act on.

This is the modern form of clinical hallucination — and the modern form of its fix.

---
## Part 3 — Classification: sorting notes into categories

**Classification** sorts a note into one of a fixed set of categories. We classify each discharge summary by what kind of follow-up it implies.

Note the prompt does two things from Block C: it **lists the exact categories**, and it gives an **`uncertain`** option so the model is not forced to guess on an ambiguous note.

In [10]:
def classify(note_text):
    prompt = f"""Classify the discharge summary below into exactly one category:
- routine: no specific follow-up, or routine GP / nurse review only
- follow-up: a specific clinic or specialist follow-up is arranged
- urgent: the patient is deteriorating or needs urgent escalation
- uncertain: the note does not give enough information to decide

Reply with only one word: routine, follow-up, urgent, or uncertain.

Discharge summary:
{note_text}"""
    return ask(prompt).strip().lower()

for n in notes:
    print(f"{n['id']}:  {classify(n['text'])}")

N01:  follow-up
N02:  routine
N03:  routine
N04:  follow-up
N05:  urgent
N06:  uncertain
N07:  routine
N08:  follow-up
N09:  follow-up
N10:  urgent
N11:  routine
N12:  follow-up


Check a few against the notes. Note **N06** (transient dizziness, follow-up "to be determined") is genuinely ambiguous — the `uncertain` category gives the model an honest answer instead of a forced guess.

In [11]:
# 🔧 YOUR TURN: classify a note of your own. Edit the text and re-run.
my_note = "Patient seen with mild ankle sprain. Advised rest and simple analgesia. No follow-up needed."

print(classify(my_note))

routine


---
## Part 4 — Zero-shot vs few-shot

So far, our `classify` function spelled out what each category means in plain English — and the model used that explanation. **Few-shot prompting earns its keep in the opposite situation: when the prompt's words can't fully tell the model what we mean, and only worked examples can.**

A realistic example: imagine your hospital tags every discharge with an internal four-code system:

- **DTC** — discharge to community (no specialist needed)
- **DTS** — discharge to specialist (clinic appointment or letter arranged)
- **URG** — urgent escalation
- **AMB** — ambiguous, needs human review

The model has never seen these codes. We try two versions of the prompt:

1. **Zero-shot** — list the codes, ask for one back. No definitions, no examples.
2. **Few-shot** — same task, plus three worked input → output pairs.

We then score both against the gold labels and read the output column carefully.

In [12]:
# Map the existing gold labels into the institutional code system.
GOLD_TO_CODE = {
    "routine":    "DTC",
    "follow-up":  "DTS",
    "urgent":     "URG",
    "uncertain":  "AMB",
}
gold_codes = [GOLD_TO_CODE[n["gold_classification"]] for n in notes]


def classify_zeroshot(note_text):
    # No definitions, no examples - only the allowed codes.
    prompt = f"""Label the discharge summary below with exactly one of these codes:
DTC, DTS, URG, AMB

Reply with only the code.

Discharge summary:
{note_text}"""
    return ask(prompt).strip()


def classify_fewshot(note_text):
    # Same codes; three worked examples teach what each one means.
    prompt = f"""Label the discharge summary below with exactly one of these codes:
DTC, DTS, URG, AMB

Examples:
Summary: "Treated for a minor viral illness. Discharged with advice. GP only if needed."
Code: DTC
Summary: "Post-operative patient. Orthopaedic clinic review arranged in six weeks."
Code: DTS
Summary: "Rapidly deteriorating; oxygen needs rising. Escalated to intensive care."
Code: URG

Reply with only the code.

Summary: "{note_text}"
Code:"""
    return ask(prompt).strip()


def normalize(reply):
    # For accuracy scoring, take just the first token, upper-cased.
    parts = reply.strip().upper().split()
    return parts[0] if parts else ""


zero = [classify_zeroshot(n["text"]) for n in notes]
few  = [classify_fewshot(n["text"]) for n in notes]

zero_acc = sum(normalize(p) == g for p, g in zip(zero, gold_codes)) / len(notes)
few_acc  = sum(normalize(p) == g for p, g in zip(few, gold_codes)) / len(notes)


def cell(s, width):
    return (s[:width-2] + "..") if len(s) > width else s.ljust(width)

print(f"{'note':6}{'gold':6}{'zero-shot (raw)':24}{'few-shot':10}")
for n, g, z, f in zip(notes, gold_codes, zero, few):
    print(f"{n['id']:6}{g:6}{cell(z, 24)}{cell(f, 10)}")
print(f"\nAccuracy - zero-shot: {zero_acc:.2f}   few-shot: {few_acc:.2f}")

note  gold  zero-shot (raw)         few-shot  
N01   DTS   DTS                     DTS       
N02   DTC   DTS                     DTC       
N03   DTC   AMB                     DTS       
N04   DTS   DTS                     DTS       
N05   URG   URG                     URG       
N06   AMB   AMB                     AMB       
N07   DTC   AMB                     DTC       
N08   DTS   DTS                     DTS       
N09   DTS   AMB                     DTS       
N10   URG   URG                     URG       
N11   DTC   AMB                     DTC       
N12   DTS   AMB                     DTS       

Accuracy - zero-shot: 0.50   few-shot: 0.92


Read the columns carefully. The zero-shot column will be noticeably worse than the few-shot one — and the *way* it is wrong is the lesson.

**The model can guess from English letters, but not from arbitrary ones.** `URG` is the one code zero-shot reliably gets right — the letters carry just enough English to hint at *urgent*. `DTC` and `DTS` both start with `DT*` and the model has no way to distinguish them from the prompt alone, so it tends to commit to one and miss the other.

**Zero-shot hedges to `AMB` when it doesn't know.** Count how often the zero-shot column lands on `AMB` for notes that are clearly *not* ambiguous in the gold column. This is what an honest-but-untrained model does when faced with a label system it has never seen: it picks the most *"I'm not sure"* looking option as a hedge. It is the model's way of saying *"don't blame me, I tried."*

**Three examples are enough to fix almost all of it.** The few-shot column rarely makes the boundary mistakes the zero-shot one made, and it almost never bails to `AMB`. Three input → output pairs taught the model the convention more cleanly than any English description could have. Some residual disagreement is normal — there are usually borderline notes where reasonable clinicians would disagree on the label. That is not a failure of few-shot; it is the irreducible noise of any human convention.

**The rule.** Few-shot prompting earns its keep when the task depends on **your conventions** — labels, codes, or output styles the model has no prior reason to know. When the prompt's words already nail the task down (as in Part 3, where every category was defined in English), examples mostly add noise. When the words *cannot* nail it down — institutional codes, domain shorthand, a custom severity scale — examples become essential.

**A short addendum on chain-of-thought.** The early-2020s prompting trick *"Let's think step by step"* mattered when models had no internal reasoning of their own. Today's frontier models — including the one we are using here — already reason internally (often called *"thinking"*), so explicit CoT prompts rarely flip an answer. What CoT still buys you is **visibility**: a clinician or auditor can read the steps and see exactly *why* the model arrived at its conclusion. In healthcare, that often matters more than the accuracy bump ever did.

---
## Part 5 — Chain-of-thought: letting the model reason

A task that needs **step-by-step reasoning**: checking a patient against several trial criteria at once.
We ask the same question twice — once demanding only a one-word answer, once asking the model to reason through each criterion first.

In [13]:
criteria_and_patient = """Clinical trial inclusion criteria:
1. Age 18-70
2. eGFR >= 60
3. No prior chemotherapy
4. Not currently pregnant

Patient: 63-year-old man. eGFR 68. Completed a course of radiotherapy in 2021.
No history of chemotherapy."""

# Version A — answer only
prompt_terse = criteria_and_patient + "\n\nIs the patient eligible? Reply with only one word: yes or no."
print("WITHOUT step-by-step reasoning:")
print(ask(prompt_terse))

WITHOUT step-by-step reasoning:
Yes


In [14]:
# Version B — reason first
prompt_cot = criteria_and_patient + """

Work through the criteria one at a time, stating whether each is met and why.
Then give your final answer: eligible or not eligible."""

print("WITH step-by-step reasoning:")
print(ask(prompt_cot))

WITH step-by-step reasoning:
To determine the patient's eligibility for the clinical trial, we will evaluate each criterion against the patient's profile:

1.  **Age 18-70:**
    *   **Patient:** 63 years old.
    *   **Status:** **Met.** (63 falls within the required range).

2.  **eGFR >= 60:**
    *   **Patient:** eGFR 68.
    *   **Status:** **Met.** (68 is greater than or equal to 60).

3.  **No prior chemotherapy:**
    *   **Patient:** No history of chemotherapy; completed radiotherapy in 2021.
    *   **Status:** **Met.** (Radiotherapy is not chemotherapy, so the patient meets the criteria of having no prior chemotherapy).

4.  **Not currently pregnant:**
    *   **Patient:** Man.
    *   **Status:** **Met.** (This criterion is not applicable to male patients).

**Final Answer:**
**Eligible.**


The reasoning version checks each criterion in turn. The key trap is criterion 3: the patient had **radiotherapy**, not **chemotherapy** — different things. For small models, a model reasoning step by step is far more likely to catch that distinction than one forced to blurt a single word.

Even when a model reaches the right answer either way, the step-by-step version has a second benefit: the reasoning is **visible, so a human can check it.**

---
## Part 6 — Structured output: JSON a program can use

So far the model replied in free text — fine for a human to read. To feed the result into other software, we want **structured JSON**.

The Gemini API can guarantee valid JSON in a shape we define. We describe the shape with a small `TypedDict` and pass it as a `response_schema`.

In [15]:
class NoteExtraction(TypedDict):
    medications: list[str]
    diagnoses: list[str]

def extract_structured(note_text):
    prompt = f"""Extract the medications and diagnoses from this discharge summary.
Use only what is explicitly stated. Generic medication names, lowercase.

{note_text}"""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=NoteExtraction,
        ),
    )
    return json.loads(response.text)

result = extract_structured(notes[0]["text"])
print("Type of result:", type(result).__name__)
print(result)

Type of result: dict
{'medications': ['aspirin', 'ticagrelor', 'atorvastatin', 'bisoprolol', 'ramipril'], 'diagnoses': ['non-st-elevation myocardial infarction']}


In [16]:
# Because it is real structured data, a program can use it directly.
print("Medications:", result["medications"])
print("Number of medications:", len(result["medications"]))
print("Diagnoses:", result["diagnoses"])

Medications: ['aspirin', 'ticagrelor', 'atorvastatin', 'bisoprolol', 'ramipril']
Number of medications: 5
Diagnoses: ['non-st-elevation myocardial infarction']


Free text is for a human to read; **JSON is for the next program to use.** This is the step that turns "a model that chats" into "a model wired into a workflow" — and it is what makes the gold-set check in the next part possible.

---
## Part 7 — Generate summaries

Here we generate a summary. Try to change the prompt to reflect different possible summarization styles and usages, and see the results.

In [18]:
note2 = [n for n in notes if n["id"] == "N02"][0]["text"]

summary_prompt = f"""Summarize the discharge summary below in two short sentences for the patient's GP.

{note2}"""
summary = ask(summary_prompt)

print("SOURCE NOTE:\n", note2)
print("\n" + "-" * 60)
print("\nMODEL SUMMARY:\n", summary)

SOURCE NOTE:
 Ms. B, 41, admitted with community-acquired pneumonia. Treated with intravenous co-amoxiclav, switched to oral after 48 hours. Afebrile and clinically improved by day three; chest X-ray showed resolving consolidation. Discharged to complete a five-day course of oral amoxicillin. Advised to see her GP only if symptoms recur. No routine follow-up required.

------------------------------------------------------------

MODEL SUMMARY:
 Ms. B was admitted with community-acquired pneumonia and showed rapid clinical improvement on intravenous followed by oral antibiotics. She has been discharged to complete a five-day course of oral amoxicillin, with no routine follow-up required.


**Compare the two carefully**, the way you did in the Block D exercise:

- Is every fact in the summary actually in the source note?
- Did the summary **add** anything — a drug, a dose, a condition the note never mentioned?
- Did it **omit** anything important?

A summary that reads fluently is not the same as a summary that is faithful. With model output, *reading fine* is not verification — **checking against the source** is.

Thankfully, modern models are very good at summarization, especially frontier's one. With smaller model, you should double check the summary quality for hallucinations.

---
## Part 8 — Checking against a gold-standard set

How do you know the model is actually doing the extraction *well* — not just plausibly? You compare its output to answers a human prepared in advance: a **gold-standard set**.

Each of our 12 notes came with `gold_medications`. We run extraction on all of them and measure the overlap.

In [19]:
def evaluate_extraction(predictions, gold_lists):
    """Micro-averaged precision and recall over all notes."""
    total_tp = total_pred = total_gold = 0
    per_note = []
    for pred, gold in zip(predictions, gold_lists):
        p = {x.strip().lower() for x in pred}
        g = {x.strip().lower() for x in gold}
        matched = len(p & g)
        total_tp += matched
        total_pred += len(p)
        total_gold += len(g)
        per_note.append((matched, len(p), len(g)))
    precision = total_tp / total_pred
    recall = total_tp / total_gold
    return precision, recall, per_note

In [20]:
# Run structured extraction on all 12 notes, then score against the gold set.
predicted_meds = [extract_structured(n["text"])["medications"] for n in notes]
gold_meds = [n["gold_medications"] for n in notes]

precision, recall, per_note = evaluate_extraction(predicted_meds, gold_meds)

print(f"{'note':6}{'matched':10}{'predicted':12}{'gold':6}")
for n, (m, p, g) in zip(notes, per_note):
    flag = "" if (m == p == g) else "  <- inspect"
    print(f"{n['id']:6}{m:<10}{p:<12}{g:<6}{flag}")

print(f"\nOverall precision: {precision:.2f}   (of what it listed, how much was correct)")
print(f"Overall recall:    {recall:.2f}   (of the gold items, how many it found)")

note  matched   predicted   gold  
N01   5         5           5     
N02   2         2           2     
N03   3         3           3     
N04   3         3           3     
N05   1         1           1     
N06   0         0           0     
N07   0         0           0     
N08   4         4           4     
N09   1         1           1     
N10   1         2           1       <- inspect
N11   1         1           1     
N12   2         2           2     

Overall precision: 0.96   (of what it listed, how much was correct)
Overall recall:    1.00   (of the gold items, how many it found)


The rows flagged `inspect` are where the model and the gold set disagree — open those notes and look. Common causes: the model named a drug more or less specifically than the gold answer, or phrased it differently ("co-amoxiclav" vs "amoxicillin-clavulanate").

That last point is the honest caveat: **real evaluation needs more than exact string matching** — it needs to handle synonyms and partial matches. This is a first taste; proper evaluation is a topic of its own on Day 2.

In [21]:
# Classification, scored the same way.
predicted_class = [classify(n["text"]) for n in notes]
gold_class = [n["gold_classification"] for n in notes]

correct = sum(p == g for p, g in zip(predicted_class, gold_class))
print(f"Classification accuracy: {correct}/{len(notes)} = {correct/len(notes):.2f}")
for n, p, g in zip(notes, predicted_class, gold_class):
    mark = "ok" if p == g else "MISS"
    print(f"  {n['id']}: predicted {p:12} gold {g:12} {mark}")

Classification accuracy: 12/12 = 1.00
  N01: predicted follow-up    gold follow-up    ok
  N02: predicted routine      gold routine      ok
  N03: predicted routine      gold routine      ok
  N04: predicted follow-up    gold follow-up    ok
  N05: predicted urgent       gold urgent       ok
  N06: predicted uncertain    gold uncertain    ok
  N07: predicted routine      gold routine      ok
  N08: predicted follow-up    gold follow-up    ok
  N09: predicted follow-up    gold follow-up    ok
  N10: predicted urgent       gold urgent       ok
  N11: predicted routine      gold routine      ok
  N12: predicted follow-up    gold follow-up    ok


---
## Part 9 — A multi-turn conversation

Until now every call was a single prompt. But you can also hold a **conversation** — the model remembers the earlier turns. This is the "conversation" stage from the Block C trajectory slide, and it is the stepping stone to **agents** later in the course.

We open a chat and ask three questions in sequence — each one builds on the last.

In [22]:
chat = client.chats.create(model=MODEL)
note1 = notes[0]["text"]

print("TURN 1 — extract")
r1 = chat.send_message(f"Here is a discharge summary:\n\n{note1}\n\nWhich medications was the patient discharged on?")
print(r1.text)

TURN 1 — extract
The patient was discharged on the following medications:

*   **Aspirin**
*   **Ticagrelor**
*   **Atorvastatin**
*   **Bisoprolol**
*   **Ramipril**


In [23]:
print("TURN 2 — a follow-up question (the model remembers the note)")
r2 = chat.send_message("Which of those medications are used to lower blood pressure?")
print(r2.text)

TURN 2 — a follow-up question (the model remembers the note)
Among the medications listed, the following are primarily used to lower blood pressure:

*   **Bisoprolol:** This is a beta-blocker, which works by slowing the heart rate and reducing the force of the heart's contraction, thereby lowering blood pressure.
*   **Ramipril:** This is an ACE (angiotensin-converting enzyme) inhibitor, which works by relaxing blood vessels, making it easier for the heart to pump blood and lowering blood pressure.

While **atorvastatin** is a statin (used to lower cholesterol) and **aspirin** and **ticagrelor** are antiplatelet agents (used to prevent blood clots), they are not considered antihypertensive medications.


In [24]:
print("TURN 3 — reasoning that builds on the conversation")
r3 = chat.send_message("The patient now reports a persistent dry cough. Which of the medications is the most likely cause?")
print(r3.text)

TURN 3 — reasoning that builds on the conversation
The most likely cause of the persistent dry cough is **Ramipril**.

A dry, persistent cough is a well-known side effect of ACE inhibitors like ramipril. It occurs in a significant number of patients due to the accumulation of bradykinin in the lungs.

**Important Note:** If the patient is experiencing this symptom, they should contact their GP or cardiology team. They should **not** stop taking the medication abruptly without medical advice, as it is an essential part of their treatment following a heart attack. A doctor will typically assess the severity of the cough and may consider switching the patient to an alternative class of medication, such as an Angiotensin II Receptor Blocker (ARB), which does not typically cause this side effect.


Notice you never re-sent the discharge summary — the model carried it forward. In turns 2 and 3 the "prompt" is no longer one block of text; it is the **whole conversation so far.**

The skill shifts from writing one perfect instruction to **steering an exchange.** And that is exactly what an agent does later in the course — except the agent will hold the conversation with *tools*, not just with you.

---
## Recap — what you did

| Part | What you practised |
|---|---|
| Extraction | Pulling structured facts from a note, with a grounding rule |
| Hallucination | Watched the model invent a detail — and a grounding rule prevent it |
| Classification | Sorting notes into categories, with an *uncertain* option |
| Zero-shot vs few-shot | Examples mainly buy consistency |
| Chain-of-thought | Step-by-step reasoning, visible and checkable |
| Structured output | JSON a program can use directly |
| Inspecting output | Checking a summary against its source |
| Gold-set check | Measuring quality against reference answers |
| Multi-turn | Steering a conversation, not just one prompt |

**You can now direct a language model on clinical text — and inspect it with a critical eye.**

**Next:** closing the two gaps you saw — giving the model the *right documents* to work from (**RAG**), and letting it *use tools and act* (**agents**).